In [1]:
import accelforge as af
from accelforge.frontend import Spec
from accelforge.mapper import Metrics

spec = Spec.from_yaml(
    af.examples.arches.simple_heterogeneous,
    af.examples.workloads.bert,
)
spec.mapper.metrics = Metrics.ENERGY | Metrics.LATENCY
spec.mapper.n_concurrent_threads = 2

pmappings = af.mapper.FFM.make_pmappings(spec, einsum_names=["Q", "K", "QK"])
mappings = af.mapper.FFM.join_pmappings(pmappings, spec.mapper.metrics, require_all_einsums=False)

Getting energy, latency, and leak power for components running QK: 100%|██████████| 3/3 [00:00<00:00, 39.55it/s]
Generating pmapping templates for compute MacA Einsum Q: 46it [00:00, 484.52it/s]
Generating pmapping templates for compute MacA Einsum K: 46it [00:00, 460.13it/s]]
Generating pmapping templates for compute MacA Einsum QK: 79it [00:00, 554.00it/s]
Generating pmapping templates for compute MacB Einsum Q: 46it [00:00, 586.32it/s]
Generating pmapping templates for compute MacB Einsum K: 46it [00:00, 587.16it/s]
Generating pmapping templates for compute MacB Einsum QK: 79it [00:00, 531.08it/s]
Generating jobs: 100%|██████████| 3/3 [00:01<00:00,  2.04it/s]


Einsum Q has 92 pmapping jobs:
	0	[WQ in MainMemory] T-b  T-m  [I in GlobalBuffer] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-b  T-d  T-e  T-h  T-m  MacA computes Q
	1	[WQ in MainMemory] T-b  T-m  [I in GlobalBuffer] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-d  T-e  T-h  [WQ in GlobalBuffer] T-b  T-m  MacA computes Q
	2	[WQ in MainMemory] T-b  T-m  [I in GlobalBuffer] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-b  T-d  T-e  T-h  T-m  MacB computes Q
	3	[WQ in MainMemory] T-b  T-m  [I in GlobalBuffer] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-d  T-e  T-h  [WQ in GlobalBuffer] T-b  T-m  MacB computes Q
	4	[WQ in MainMemory] T-b  T-m  [Q in GlobalBuffer] T-b  T-d  T-m  [I in GlobalBuffer] T-b  T-d  T-e  T-h  T-m  MacA computes Q
	5	[WQ in MainMemory] T-b  T-m  [Q in GlobalBuffer] T-b  T-d  T-m  [I in GlobalBuffer] T-d  T-e  T-h  [WQ in GlobalBuffer] T-b  T-m  MacA computes Q
	6	[WQ in MainMemory] T-b  T-m  [Q in GlobalBuffer] T-b  T-d  T-m  [I in GlobalBuffer] T-b  T-d  T-e  T-h  T-m  MacB c

Grouping pmappings for QK: 100%|██████████| 270/270 [00:00<00:00, 482.22it/s]


Q: 1.53e05 total, 3.88e04 (1/4) valid, 3.88e04 (1/4) evaluated, 3.07e03 (1/50) Pareto-Optimal
K: 1.53e05 total, 3.88e04 (1/4) valid, 3.88e04 (1/4) evaluated, 3.07e03 (1/50) Pareto-Optimal
QK: 7.24e05 total, 1.35e05 (1/5) valid, 1.40e05 (1/5) evaluated, 2.12e04 (1/34) Pareto-Optimal
Total: 1.03e06 total, 2.13e05 (1/5) valid, 2.17e05 (1/5) evaluated, 2.73e04 (1/38) Pareto-Optimal


Compressing pmappings: 100%|██████████| 3/3 [00:02<00:00,  1.43it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>latency=4.03e+08    latency<SEP>memory<SEP>GlobalBuffer=0.00e+00    latency<SEP>memory<SEP>MainMemory=0.00e+00    Total<SEP>energy=1.33e+10
Final clean join.


Dirty pruning pmappings: 100%|██████████| 368/368 [00:00<00:00, 3240.44it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 27322 -> 27322 (100.00% kept) pmappings


Grouping pmappings: 100%|██████████| 1/1 [00:00<00:00, 30.11it/s]

Dirty joining mapping(s) valid & optimal! Returning...


In [ ]:
from copy import copy, deepcopy
from uuid import uuid4

import pandas as pd

from accelforge.frontend import arch
from accelforge.frontend.arch import Memory
from accelforge.frontend.mapping.mapping import MappingNodeWithChildren
from accelforge.frontend.renames import EinsumName, TensorName
from accelforge.frontend.spec import Mapping, Spec
from accelforge.util._frozenset import oset
from accelforge.frontend.mapping import (
    Compute,
    Reservation,
    Split,
    Nested,
    NodeList,
    TensorHolder,
)
from accelforge.frontend.workload import Workload
from accelforge.frontend._workload_isl._symbolic import (
    get_stride_and_halo,
    get_rank_variable_relevancy,
)
from accelforge.mapper.FFM._make_pmappings.make_pmappings_from_templates.symbol_relations import (
    get_initial_delta_choices,
)
from accelforge.mapper.FFM._pareto_df.df_convention import col_used_in_joining


def evaluate_mapping(
    spec: Spec,
    flattened_arches: dict[(EinsumName, str), list[arch.Leaf]] | None = None,
    evaluated_specs: dict[EinsumName, Spec] | None = None,
):
    """
    Evaluate a mapping.

    Parameters
    ----------
    spec:
        The specification of architecture, workload, and mapping.
    flattened_arches:
        A dictionary of (EinsumName, Compute Name) to lists of architecture nodes. These
        contain the evaluated and flattened architecture node for that particular Einsum
        and compute combination. If provided, then these will be used instead of
        re-parsing the architecture.
    evaluated_specs:
        A dictionary of Einsum names to evaluated specifications. These contain the evaluated
        specification for that particular Einsum. If provided, then these will be used
        instead of re-parsing the specification.
    """
    from accelforge.mapper.FFM._join_pmappings.compatibility import Compatibility
    from accelforge.mapper.FFM._join_pmappings.pmapping_dataframe import (
        PmappingDataframe,
    )
    from accelforge.mapper.FFM._join_pmappings.pmapping_group import PmappingGroup
    from accelforge.mapper.FFM._join_pmappings.join_pmappings import (
        clean_compress_and_join_pmappings,
    )
    from accelforge.mapper.FFM.pmappings import MultiEinsumPmappings
    from accelforge.mapper.FFM._make_pmappings.make_pmappings import (
        get_rank_variable_bounds_for_all_einsums,
    )
    from accelforge.model.run_model import (
        run_model,
    )
    from accelforge.mapper.FFM._make_pmappings.make_pmappings_from_templates.make_tile_shapes import (
        _calculate_iterations_and_rank_columns,
        _clean_energy_columns,
    )
    from accelforge.mapper.FFM._make_pmappings.pmapper_job import Job

    assert (evaluated_specs is not None) == (
        flattened_arches is not None
    ), f"Provide either flattened_arches or evaluated_specs, not both."

    original_job = Job(
        metrics=spec.model.metrics,
        rank_variable_bounds=get_rank_variable_bounds_for_all_einsums(spec),
        spec_one_einsum=spec,
        resource_usage_tolerance=0,  # spec.model.resource_usage_tolerance,
        objective_tolerance=0,  # spec.model.objective_tolerance,
        workload_n_einsums=len(spec.workload.einsum_names),
    )

    einsum2pmappings = {}
    pmapping_objects = {}
    einsum2jobs = {}
    s = (
        "Spec must not be evaluated before evaluating a mapping. Was "
        "this spec returned by spec.calculate_component_area_energy_latency_leak()?"
    )

    needs_reservations = not bool(spec.mapping.get_nodes_of_type(Reservation))

    fusable_tensors = spec.workload.tensor_names_used_in_multiple_einsums
    stride_and_halo = get_stride_and_halo(spec.workload)

    assert not getattr(spec, "_evaluated", False), s
    for pmapping in _split_mapping_to_pmappings(spec.mapping, spec.workload):
        einsum_name = pmapping.nodes[-1].einsum
        compute_name = pmapping.nodes[-1].component
        pmapping_id = uuid4()
        job = copy(original_job)

        if flattened_arches is not None:
            flattened_arch = flattened_arches[(einsum_name, compute_name)]
            cur_spec = evaluated_specs[einsum_name]

        else:
            cur_spec = spec.calculate_component_area_energy_latency_leak(
                einsum_name=einsum_name,
                area=False,
            )
            flattened_arch = cur_spec._get_flattened_architecture(
                compute_node=pmapping.nodes[-1].component
            )

        job.spec_one_einsum = cur_spec
        job.einsum_name = pmapping.nodes[-1].einsum
        job.stride_and_halo = stride_and_halo
        # spec, not cur_spec, becuase cur_spec only has one einsum and the delta choices
        # depend on >1 Einsums
        job.initial_delta_choices = get_initial_delta_choices(
            job.einsum_name, spec.workload
        )
        pmapping.split_reservations()
        pmapping.split_loop_with_multiple_rank_variables(job.einsum_name)
        pmapping.split_tensor_holders_with_multiple_tensors()
        _add_backing_to_tensor_holders(pmapping)

        job.mapping = pmapping
        job.tensor_to_relevancy = {
            tensor: get_rank_variable_relevancy(
                job.spec_one_einsum.workload.einsums[job.einsum_name], tensor
            )
            for tensor in job.spec_one_einsum.workload.einsums[
                job.einsum_name
            ].tensor_names
        }
        pmapping.clear_irrelevant_reservations(oset(job.tensor_to_relevancy))

        einsum2jobs[job.einsum_name] = job

        job.flattened_arch = flattened_arch
        job.memories_track_all = [
            m.name for m in flattened_arch if isinstance(m, Memory)
        ]

        job.fusable_tensors = fusable_tensors & oset(job.tensor_to_relevancy)
        einsum = cur_spec.workload.einsums[job.einsum_name]

        _, df, _, _, tensor2mapping, _ = run_model(
            job, add_reservations=needs_reservations
        )

        # Calculate iteration counts and rank columns
        _clean_energy_columns(df, job.metrics)
        _calculate_iterations_and_rank_columns(
            job.mapping.nodes, job, df, job.rank_variable_bounds
        )
        compatibility = Compatibility.from_mapping(
            job.mapping,
            job.fusable_tensors,
            einsum,
        )
        symbol_renames, compatibility = compatibility.make_fused_loop_symbols(
            einsum_name
        )
        for k, v in symbol_renames.items():
            df[v] = df.pop(k)

        new_df = {}
        for key, value in df.items():
            if not col_used_in_joining(key):
                key = f"{job.einsum_name}<SEP>{key}"
            # Want usage both for joining & for per-einsum info
            if key.startswith("usage<SEP>"):
                new_df[f"{job.einsum_name}<SEP>{key}"] = value
            new_df[key] = value
        df = new_df
        df[f"{job.einsum_name}<SEP>mapping"] = pmapping_id

        einsum2pmappings[job.einsum_name] = [
            PmappingGroup(
                compatibility,
                PmappingDataframe(
                    data=pd.DataFrame(df, columns=df.keys(), index=[0]),
                    n_total_pmappings=1,
                    n_valid_pmappings=1,
                    ignored_resources=oset(),
                    drop_valid_reservations=False,
                    n_concurrent_threads=spec.mapper.n_concurrent_threads
                ),
            )
        ]
        pmapping_objects[job.einsum_name] = {pmapping_id: job.mapping}

    # Restore the original order
    einsum2pmappings = {
        einsum_name: einsum2pmappings[einsum_name]
        for einsum_name in spec.workload.einsum_names
        if einsum_name in einsum2pmappings
    }
    # return MultiEinsumPmappings(
    #     spec=spec,
    #     einsum2pmappings=einsum2pmappings,
    #     pmapping_objects=pmapping_objects,
    #     einsum2jobs=einsum2jobs,
    #     can_combine_multiple_runs=False,
    #     einsums_with_pmappings_generated=oset(spec.workload.einsum_names),
    #     flattened_arches=flattened_arches,
    #     evaluated_specs=evaluated_specs,
    # )
    # return einsum2pmappings

    return clean_compress_and_join_pmappings(
        pmappings=MultiEinsumPmappings(
            spec=spec,
            einsum2pmappings=einsum2pmappings,
            pmapping_objects=pmapping_objects,
            einsum2jobs=einsum2jobs,
            can_combine_multiple_runs=False,
            einsums_with_pmappings_generated=oset(spec.workload.einsum_names),
            flattened_arches=flattened_arches,
            evaluated_specs=evaluated_specs,
        ),
        metrics=spec.model.metrics,
        print_progress=False,
        for_model=True,
    )


def _add_backing_to_tensor_holders(pmapping: Mapping):
    seen_tensors = oset()
    for node in pmapping.nodes:
        if isinstance(node, TensorHolder):
            new_tensors = oset(node.tensors) - seen_tensors
            node._backing = new_tensors
            seen_tensors.update(new_tensors)


def _split_mapping_worker(node: MappingNodeWithChildren):
    if isinstance(node, Split):
        for subnodes in node.nodes:
            yield from _split_mapping_worker(subnodes)
        return

    assert isinstance(node, Nested), "BUG"

    for n in node.nodes[:-1]:
        assert not isinstance(n, MappingNodeWithChildren), "BUG"

    if not isinstance(node.nodes[-1], MappingNodeWithChildren):
        yield node.nodes
        return

    for subnodes in _split_mapping_worker(node.nodes[-1]):
        yield node.nodes[:-1] + subnodes


def _split_mapping_to_pmappings(mapping: Mapping, workload: Workload):
    """
    A DFS-like algorithm to split a mapping into pmappings at Split nodes.

    DFS has to be modified because the tree has list of nodes for nested nodes
    instead of links to children.
    """
    for nodes in _split_mapping_worker(mapping):
        mapping = Mapping(nodes=deepcopy(nodes))
        _remove_storage_of_unrelevant_tensors(mapping, workload)
        yield mapping


def _remove_storage_of_unrelevant_tensors(pmapping: Mapping, workload: Workload):
    """
    Remove tensors from Storage nodes that are not relevant to the Einsum being
    mapped.
    """
    einsum_name = pmapping.nodes[-1].einsum
    einsum = workload.einsums[einsum_name]
    relevant_tensors = oset(t.name for t in einsum.tensor_accesses)

    new_nodes = []
    for node in pmapping.nodes:
        if isinstance(node, TensorHolder):
            node.tensors = [t for t in node.tensors if t in relevant_tensors]
            if node.tensors:
                new_nodes.append(node)
        else:
            new_nodes.append(node)

    pmapping.nodes = new_nodes


In [5]:
for i, p in enumerate(pmappings.einsum2pmappings["Q"]):
    if not p.compatibility.n_loops == 3:
        continue
    print(i, p.compatibility)

10 Compatibility(n_loops=3, tensors={Reservation('I', (), 'MainMemory'), Reservation('Q', (Loop('E', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>1), False), Loop('H', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>2), False), Loop('M', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>3), False)), 'GlobalBuffer')}, splits={}, reservation_indices={0, 3})
18 Compatibility(n_loops=3, tensors={Reservation('I', (Loop('M', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>3), False),), 'GlobalBuffer'), Reservation('Q', (Loop('M', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>3), False), Loop('E', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>5), False), Loop('H', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>6), False)), 'GlobalBuffer')}, splits={}, reservation_indices={0, 1, 3})
21 Compatibility(n_loops=3, tensors={Reservation('I', (), 'MainMemory'), Reservation('Q', (Loop('M', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>5), False), Loop('E', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>6), False), Loop

In [6]:
local_spec = deepcopy(spec)
local_spec.model.metrics = local_spec.mapper.info_metrics
local_spec.mapping = mappings.data.iloc[0]["Total<SEP>mapping"]()

this_pmapping = evaluate_mapping(
    local_spec,
    flattened_arches=mappings.flattened_arches,
    evaluated_specs=mappings.evaluated_specs,
)

In [7]:

this_pmapping.einsum2pmappings["Q"][0].compatibility

Compatibility(n_loops=3, tensors={Reservation('I', (), 'MainMemory'), Reservation('Q', (Loop('E', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>0) tile_shape=1, False), Loop('H', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>1) tile_shape=1, False), Loop('M', in [0..fused_loop<SEP>Q<SEP>n_iterations<SEP>2) tile_shape=1, False)), 'GlobalBuffer')}, splits={}, reservation_indices={0, 3})

In [8]:
list(this_pmapping.einsum2pmappings["Q"][0].mappings.data.columns)

['tensor<SEP>I',
 'Q<SEP>usage<SEP>memory<SEP>MainMemory<SEP>I',
 'tensor<SEP>Q',
 'Q<SEP>usage<SEP>memory<SEP>GlobalBuffer<SEP>Q',
 'Q<SEP>usage<SEP>memory<SEP>MainMemory<SEP>WQ',
 'reservation<SEP>MainMemory<SEP>0<SEP>right<SEP>0',
 'reservation<SEP>GlobalBuffer<SEP>3<SEP>right<SEP>0',
 'Q<SEP>action<SEP>MainMemory<SEP>I<SEP>read',
 'Q<SEP>action<SEP>MainMemory<SEP>I<SEP>write',
 'Q<SEP>action<SEP>GlobalBuffer<SEP>Q<SEP>read',
 'Q<SEP>action<SEP>GlobalBuffer<SEP>Q<SEP>write',
 'Q<SEP>action<SEP>MainMemory<SEP>WQ<SEP>read',
 'Q<SEP>action<SEP>MainMemory<SEP>WQ<SEP>write',
 'Q<SEP>action<SEP>MacA<SEP>None<SEP>compute',
 'Q<SEP>energy<SEP>MainMemory<SEP>I<SEP>read',
 'Q<SEP>energy<SEP>GlobalBuffer<SEP>Q<SEP>read',
 'Q<SEP>energy<SEP>GlobalBuffer<SEP>Q<SEP>write',
 'Q<SEP>energy<SEP>MainMemory<SEP>WQ<SEP>read',
 'Q<SEP>energy<SEP>MacA<SEP>None<SEP>compute',
 'Q<SEP>energy<SEP>MainMemory<SEP>leak',
 'Q<SEP>energy<SEP>GlobalBuffer<SEP>leak',
 'Q<SEP>energy<SEP>MacA<SEP>leak',
 'Q<SEP>energ

In [9]:
for pmapping in _split_mapping_to_pmappings(mappings.mapping(), spec.workload):
    print(pmapping)

nodes=[Storage(tensors=['K', 'I'], component='MainMemory', component_object=Memory(name='MainMemory', spatial=[], component_class=None, component_model=None, component_modeling_log=['Calculating energy for MainMemory action read.', 'Setting MainMemory energy to action.energy=1', 'Calculating energy for MainMemory action write.', 'Setting MainMemory energy to action.energy=1', 'Calculating latency for MainMemory action read.', 'Setting MainMemory latency to action.latency=0', 'Calculating latency for MainMemory action write.', 'Setting MainMemory latency to action.latency=0', 'Using predefined leak power value self.leak_power=0', 'Calculating energy for MainMemory action read.', 'Setting MainMemory energy to action.energy=1', 'Calculating energy for MainMemory action write.', 'Setting MainMemory energy to action.energy=1', 'Calculating latency for MainMemory action read.', 'Setting MainMemory latency to action.latency=0', 'Calculating latency for MainMemory action write.', 'Setting Main

In [10]:
mappings.data

,tensor<SEP>I,tensor<SEP>K,Total<SEP>latency,latency<SEP>memory<SEP>GlobalBuffer,latency<SEP>memory<SEP>MainMemory,Total<SEP>energy,tensor<SEP>Q,tensor<SEP>QK,K<SEP>stride1,K<SEP>stride2,...,QK<SEP>stride5,QK<SEP>n_iterations<SEP>0,QK<SEP>n_iterations<SEP>1,QK<SEP>n_iterations<SEP>2,QK<SEP>n_iterations<SEP>3,QK<SEP>n_iterations<SEP>4,QK<SEP>n_iterations<SEP>5,QK<SEP>mapping,QK<SEP>stride6,Total<SEP>mapping
0,0,0,402653184.0,0.0,0.0,1.326658e+10,0,0,0.0,0.0,...,0.0,1.0,64.0,8.0,512.0,0.0,0.0,[nodes],0.0,<accelforge.mapper.FFM._join_pmappings.join_pm...


In [11]:
list(mappings.data.columns)

['tensor<SEP>I',
 'tensor<SEP>K',
 'Total<SEP>latency',
 'latency<SEP>memory<SEP>GlobalBuffer',
 'latency<SEP>memory<SEP>MainMemory',
 'Total<SEP>energy',
 'tensor<SEP>Q',
 'tensor<SEP>QK',
 'K<SEP>stride1',
 'K<SEP>stride2',
 'K<SEP>stride3',
 'K<SEP>mapping',
 'K<SEP>stride0',
 'Q<SEP>stride1',
 'Q<SEP>stride2',
 'Q<SEP>stride3',
 'Q<SEP>stride5',
 'Q<SEP>stride6',
 'Q<SEP>stride7',
 'Q<SEP>n_iterations<SEP>0',
 'Q<SEP>n_iterations<SEP>1',
 'Q<SEP>n_iterations<SEP>2',
 'Q<SEP>n_iterations<SEP>3',
 'Q<SEP>n_iterations<SEP>4',
 'Q<SEP>n_iterations<SEP>5',
 'Q<SEP>n_iterations<SEP>6',
 'Q<SEP>n_iterations<SEP>7',
 'Q<SEP>mapping',
 'Q<SEP>stride4',
 'QK<SEP>stride1',
 'QK<SEP>stride2',
 'QK<SEP>stride3',
 'QK<SEP>stride4',
 'QK<SEP>stride5',
 'QK<SEP>n_iterations<SEP>0',
 'QK<SEP>n_iterations<SEP>1',
 'QK<SEP>n_iterations<SEP>2',
 'QK<SEP>n_iterations<SEP>3',
 'QK<SEP>n_iterations<SEP>4',
 'QK<SEP>n_iterations<SEP>5',
 'QK<SEP>mapping',
 'QK<SEP>stride6',
 'Total<SEP>mapping']